# Task 2.1 — Dataset Selection and Setup
**Paper:** Algorithms for Learning Kernels Based on Centered Alignment  
**Student:** Pintu Singh | Roll No: 230105

## Dataset Choice and Justification

**Dataset:** `make_classification` from scikit-learn — a synthetic binary classification dataset with 300 samples, 15 features (8 informative, 3 redundant, 4 noise), generated with `random_state=42`.

**Why it is a reasonable testbed:**  
The paper's core method (ALIGNF) learns kernel weights for binary SVM classification. This synthetic dataset provides exactly that setting: binary labels, continuous numeric features, and a mix of informative and redundant features — which creates a realistic scenario where some base kernels (e.g., RBF with the right bandwidth) will align better with the target than others (e.g., linear on noisy features), giving ALIGNF a meaningful selection task.

**Why it is appropriate for this specific paper (not just SVMs in general):**  
The ALIGNF algorithm's value becomes visible only when base kernels differ in quality. The redundant and noise features in this dataset mean that a polynomial kernel (which amplifies noise dimensions) will have lower centered alignment with the target than an RBF kernel that captures local structure — exactly the diversity the paper needs to demonstrate differential kernel weighting.

**Limitations compared to the original paper:**  
The paper uses real-world UCI datasets (e.g., Ionosphere, Breast Cancer, Adult) where the relationship between features and labels is unknown and kernels represent genuine domain knowledge. Here, the data-generating process is known and controlled, which means the results are less ecologically valid. Additionally, with only 300 samples, we cannot test the paper's computational scalability claims.

**Base kernels used (following paper's Section 5.1):**  
- Linear kernel  
- RBF kernel (γ = 0.01)  
- RBF kernel (γ = 0.1)  
- Polynomial kernel (degree 2)  
- Polynomial kernel (degree 3)

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Generate dataset
X, y = make_classification(
    n_samples=300,
    n_features=15,
    n_informative=8,
    n_redundant=3,
    n_classes=2,
    random_state=42
)

# Convert labels to {-1, +1} as required by SVM and kernel alignment
y_svm = 2 * y - 1

# Train-test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_svm, test_size=0.3, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Features:         {X_train.shape[1]}")
print(f"Label distribution (train): +1={np.sum(y_train==1)}, -1={np.sum(y_train==-1)}")

Training samples: 210
Test samples:     90
Features:         15
Label distribution (train): +1=99, -1=111


## Preprocessing
The `make_classification` dataset is already normalized (zero mean, unit variance per feature by default). No additional scaling is applied. Labels are converted from {0, 1} to {-1, +1} as required by the SVM formulation and the target kernel $K_y = yy^\top$ used in the ALIGNF alignment computation.